In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.preprocessing import BRANDS_MAPPING, FA_BRANDS_MAPPING, BRAND_FA_TO_MODEL_MAPPING
from src.preprocessing import transform_models, TRANSMISSIONS_MAPPING, FUEL_MAPPING
from src.preprocessing import LOC_FA_TO_ENG, CITY_TO_PROVINCE, COLOR_FA_TO_ENG, BODY_STATUS_FA_TO_ENG
from src.preprocessing import TRIM_FA_TO_ENG

In [4]:
def load_data(file_path: str = "../data/raw/bama_backup.csv") -> pd.DataFrame:
    # Read the raw data from the specified file path.
    df = pd.read_csv(file_path, encoding="utf-8")

	# flag org columns
    df.rename(columns={
        'title': 'title_org',
        'subtitle': 'subtitle_org',
        'trim': 'trim_org',
        'year': 'year_org',
        'mileage': 'mileage_org',
        'location': 'location_org',
        'transmission': 'transmission_org',
        'fuel': 'fuel_org',
        'body_color': 'body_color_org',
        'inside_color': 'inside_color_org',
        'body_status': 'body_status_org',
        'body_type': 'body_type_org',
        'cylinder_fa': 'cylinder_fa_org',
        'image_count': 'image_count_org',
        'modified_date': 'modified_date_org',
        'brand': 'brand_org',
        'price': 'price_org',
        'link': 'link_org',
        'scraped_at': 'scraped_at_org'
	},
    inplace=True
	)

	# add listing id
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'listing_id'}, inplace=True)
    df['listing_id'] = df['listing_id'] + 1

    # fix brand names
    df['brand_clean'] = df.loc[:, 'brand_org'].str.strip().str.lower().replace(BRANDS_MAPPING).str.upper()

	# extract the model name
    df_title = df['title_org'].str.split('،', expand=True).rename(
    columns={0: 'brand_org', 1: 'model_org'}
	)
    df_title['brand'] = df_title['brand_org'].replace(FA_BRANDS_MAPPING)
    df_title['model_p1'] = df_title['brand_org'].replace(BRAND_FA_TO_MODEL_MAPPING)

    df_model = transform_models(df_title)

	# fix remaining model names manually
    df_model.loc[4178, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[6058, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[13654, 'model_org_clean'] = "Irankhodro Samand LX"
    df_model.loc[14376, 'model_org_clean'] = "Saipagroup Pride SD"
    df_model.loc[14875, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[15942, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[16513, 'model_org_clean'] = "Irankhodro Dena Plus"
    df_model.loc[16879, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[22266, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[27774, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[29193, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[30790, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[32865, 'model_org_clean'] = "Dongfeng H30 Cross"
    df_model.loc[35247, 'model_org_clean'] = "Saipagroup Atlas Manual"
    df_model.loc[35266, 'model_org_clean'] = "Irankhodro Pride"

	# add model name
    df['model_clean'] = df_model['model_p1'] + " " + df_model['model_org_clean']
    df['model_clean'] = df['model_clean'].str.strip()

	# fix trim data
    df['trim_clean'] = df['trim_org'].map(TRIM_FA_TO_ENG).str.strip()

	# fix dates data types
    df['modified_date_clean'] = pd.to_datetime(df.loc[:, 'modified_date_org'].str.slice(0, 10))
    df['scraped_at_clean'] = pd.to_datetime(df.loc[:, 'scraped_at_org'])

	# fix mileage data
    df['kilometer_e-03_clean'] = (df['mileage_org'] / 1e03).astype(float) # make it scientific notation
    df['is_zero_kilometer'] = df['kilometer_e-03_clean'] == 0

	# fix transmission data
    df['transmission_clean'] = df['transmission_org'].map(TRANSMISSIONS_MAPPING).str.strip()

	# fix fuel data
    df['fuel_clean'] = df['fuel_org'].map(FUEL_MAPPING).str.strip()

	# fix location data
    df['city_clean'] = df['location_org'].str.split(
    '/', expand=True
	)[0].map(LOC_FA_TO_ENG).str.strip()
    df['province_clean'] = df['city_clean'].map(CITY_TO_PROVINCE).str.strip()

	# fix color data
    df['body_color_clean'] = df['body_color_org'].map(COLOR_FA_TO_ENG).str.strip()
    df['inside_color_clean'] = df['inside_color_org'].map(COLOR_FA_TO_ENG).str.strip()

	# fix body status data
    df['body_status_clean'] = df['body_status_org'].map(BODY_STATUS_FA_TO_ENG).str.strip()

	# fix body type data
    df['body_type_clean'] = df['body_type_org'].str.strip().str.title()

	# handle missing values
    # fuel column
    df.loc[35618, 'fuel_org'] = 'بنزینی'
    df.loc[38383, 'fuel_org'] = 'بنزینی'
    df.loc[38403, 'fuel_org'] = 'بنزینی'
    df.loc[38998, 'fuel_org'] = 'بنزینی'

	# cylinder_fa column
    # df_missing = df.loc[df['cylinder_fa_org'].isnull()]
    # df_missing.loc[
	# 	df_missing['fuel_org'] == 'برقی', 'cylinder_fa_org'
	# ] = "0"

	# df_missing.loc[df_missing['fuel'] == 'بنزینی', 'title']

	# rearange columns
    df_clean = df.loc[
        :,
        [
            "listing_id",
            "brand_clean",
            "model_clean",
            "trim_clean",
            "year_org",
            "modified_date_clean",
            "scraped_at_clean",
            "kilometer_e-03_clean",
            "is_zero_kilometer",
            "transmission_clean",
            "fuel_clean",
            "city_clean",
            "province_clean",
            "body_color_clean",
            "inside_color_clean",
            "body_status_clean",
            "body_type_clean",
            "image_count_org",
            "cylinder_fa_org",
            "price_org"
		]
	]
    
    return df, df_clean

df, df_clean = load_data()
df

,listing_id,title_org,subtitle_org,trim_org,year_org,mileage_org,location_org,transmission_org,fuel_org,body_color_org,...,kilometer_e-03_clean,is_zero_kilometer,transmission_clean,fuel_clean,city_clean,province_clean,body_color_clean,inside_color_clean,body_status_clean,body_type_clean
0,1,سیتروئن، C3 XR,1404 | تیپ V1,تیپ V1,1404,0,تهران / ستارخان,اتوماتیک,بنزینی,مشکی,...,0.0,True,Automatic,ICE,Tehran,Tehran,Black,Black,Original Paint,Crossover
1,2,لوکانو، L7,1405 | 1.6 لیتر توربو,1.6 لیتر توربو,1405,0,تهران / میرداماد,اتوماتیک,بنزینی,سفید,...,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Crossover
2,3,هیوندای، النترا,2025 | 1.5 لیتر,1.5 لیتر,2025,0,تهران / ستارخان,اتوماتیک,بنزینی,سفید,...,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Passenger_Car
3,4,چانگان، CS 55 پلاس,2024 | 1.5 لیتر توربو,1.5 لیتر توربو,2024,28000,تهران / ولنجک,اتوماتیک,بنزینی,خاکستری,...,28.0,False,Automatic,ICE,Tehran,Tehran,Gray,Black,Original Paint,Crossover
4,5,کی ام سی، T8,1403,دنده ای,1403,30000,تهران / میرداماد,دنده ای,بنزینی,مشکی,...,30.0,False,Manual,ICE,Tehran,Tehran,Black,Black,Original Paint,Pickup
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39656,39657,ب ام و، سری 5 سدان,2015 | 528i,528i,2015,77000,اصفهان / سعادت آباد,اتوماتیک,بنزینی,سفید,...,77.0,False,Automatic,ICE,Isfahan,Isfahan,White,Maroon,Original Paint,Passenger_Car
39657,39658,ب ام و، سری 5 سدان,2018 | 530i,530i,2018,49000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,مشکی,...,49.0,False,Automatic,ICE,Isfahan,Isfahan,Black,Mocha,Original Paint,Passenger_Car
39658,39659,نیسان، ترا,1404 | اکسکلوسیو,اکسکلوسیو,1404,0,اصفهان / سعادت آباد,اتوماتیک,بنزینی,مشکی,...,0.0,True,Automatic,ICE,Isfahan,Isfahan,Black,Maroon,Original Paint,Suv
39659,39660,ب ام و، سری 7,2017 | 730li,730li,2017,42000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,سفید صدفی,...,42.0,False,Automatic,ICE,Isfahan,Isfahan,Pearl White,Maroon,Original Paint,Passenger_Car


In [55]:
df['trim_org'].value_counts()

trim_org
دنده ای                      4487
اتوماتیک                     3696
1.5 لیتر توربو               1690
تیپ 2                        1533
2.0 لیتر                      961
                             ... 
1.5 لیتر مکس اسپرت لاکچری       1
640i                            1
2.0 لیتر تيپ 2                  1
2.3 لیتر پرستیژ                 1
فایتر (توربو)                   1
Name: count, Length: 598, dtype: int64

In [3]:
# unique cities
df['location_org'].str.split('/', expand=True)[0].unique()

<StringArray>
[   'تهران ',    'شیراز ',    'شهریار',   'پل دختر',     'زنجان',     'همدان',
     'مشهد ',   'اصفهان ',     'تبریز',      'ابهر',
 ...
    'آشتیان',   'عجب شیر',    'بوانات',      'راور',    'رامیان',      'کیار',
    'زرندیه', 'قائم شهر ',   'مهاباد ', 'شاهین دژ ']
Length: 374, dtype: str

In [4]:
# unique values for transmission
df['transmission_org'].value_counts()

transmission_org
اتوماتیک    24036
دنده ای     15625
Name: count, dtype: int64

In [5]:
df['fuel_org'].value_counts()

fuel_org
بنزینی           33811
هیبریدی           2011
برقی              1551
دوگانه سوز         844
پلاگین هیبرید      677
هیبرید ملایم       371
بردافزا            348
دیزلی               48
Name: count, dtype: int64

In [6]:
df['body_color_org'].unique()

<StringArray>
[      'مشکی',       'سفید',    'خاکستری',    'نقره ای',   'آلبالویی',
   'تیتانیوم',    'سرمه ای',          '-',         'بژ',        'آبی',
       'قرمز',  'نوک مدادی',     'نقرآبی',  'سفید صدفی',        'کرم',
       'یشمی',   'بادمجانی',        'مسی',    'قهوه ای',       'طوسی',
        'زرد',      'ذغالی',        'سبز',     'دلفینی',   'کربن بلک',
      'زرشکی',      'اطلسی',     'زیتونی',     'نارنجی',       'موکا',
       'بنفش',       'خاکی',       'شتری',       'سربی',     'گیلاسی',
       'عدسی',       'یاسی',     'اخرائی',      'عنابی',      'طلائی',
      'صورتی',       'برنز',      'مارون',   'امبر بلک',    'تارتوفو',
 'پوست پیازی']
Length: 46, dtype: str

In [7]:
df['inside_color_org'].unique()

<StringArray>
[      'مشکی',          '-',      'مارون',       'طوسی',    'خاکستری',
         'بژ',       'قرمز',       'سفید',        'کرم',    'نقره ای',
    'تارتوفو',  'نوک مدادی',    'سرمه ای',   'آلبالویی',    'قهوه ای',
       'شتری',     'نارنجی',       'موکا',   'امبر بلک',  'سفید صدفی',
        'مسی',        'آبی',      'عنابی',   'بادمجانی',   'کربن بلک',
      'طلائی',     'دلفینی',   'تیتانیوم',     'اخرائی',     'گیلاسی',
       'یشمی',       'برنز',       'بنفش',      'زرشکی',      'ذغالی',
     'زیتونی',        'زرد',       'عدسی',       'خاکی',      'اطلسی',
     'نقرآبی',        'سبز',       'سربی', 'پوست پیازی',      'صورتی',
       'یاسی']
Length: 46, dtype: str

In [8]:
df.loc[:, 'body_status_org'].value_counts()

body_status_org
بدون رنگ            28191
چند لکه رنگ          2053
یک لکه رنگ           1799
دور رنگ              1708
دو لکه رنگ           1620
خط و خش جزئی         1119
صافکاری بدون رنگ      680
کامل رنگ              611
گلگیر رنگ             464
گلگیر تعویض           277
کاپوت تعویض           266
کاپوت رنگ             219
یک درب رنگ            173
دو درب رنگ            146
درب تعویض             143
تصادفی                110
اتاق تعویض             56
سوخته                  21
اوراقی                  5
Name: count, dtype: int64

In [9]:
df[df['inside_color_clean'] == "Unknown"]

,listing_id,title_org,subtitle_org,trim_org,year_org,mileage_org,location_org,transmission_org,fuel_org,body_color_org,...,scraped_at_clean,kilometer_e-03_clean,is_zero_kilometer,transmission_clean,fuel_clean,city_clean,province_clean,body_color_clean,inside_color_clean,body_status_clean
5,5,فیدلیتی، الیت,1404 | تیپ 1 پنج نفره,تیپ 1 پنج نفره,1404,20000,شیراز / آرامستان دارالرحمه,اتوماتیک,بنزینی,مشکی,...,2026-10-14,20.0,False,Automatic,ICE,Shiraz,Fars,Black,Unknown,Original Paint
24,24,میتسوبیشی، اوتلندر,2023 | تک دیفرانسیل H لاین,تک دیفرانسیل H لاین,2023,33000,تهران / اقدسیه,اتوماتیک,هیبرید ملایم,سفید,...,2026-10-14,33.0,False,Automatic,MHEV,Tehran,Tehran,White,Unknown,Original Paint
36,36,فولکس، ID.4 Crozz,2025 | پیور,پیور,2025,14000,تهران / فرمانیه,اتوماتیک,برقی,سرمه ای,...,2026-10-14,14.0,False,Automatic,EV,Tehran,Tehran,Navy Blue,Unknown,Original Paint
37,37,ب ام و، X4,2017 | 28,28,2017,33000,اصفهان / احمدآباد,اتوماتیک,بنزینی,مشکی,...,2026-10-14,33.0,False,Automatic,ICE,Isfahan,Isfahan,Black,Unknown,Original Paint
40,40,زامیاد، پادرا پلاس,1405 | دوگانه سوز,دوگانه سوز,1405,0,تهران / امیرآباد,دنده ای,دوگانه سوز,سفید,...,2026-10-14,0.0,True,Manual,Bi-Fuel,Tehran,Tehran,White,Unknown,Original Paint
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39456,39456,آرتابان (ای ام جی)، توکا,1405 | ساده,ساده,1405,0,تهران / زعفرانیه,اتوماتیک,بنزینی,نقره ای,...,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,Silver,Unknown,Original Paint
39469,39469,لوکانو، L8,1405 | 2.0 لیتر توربو,2.0 لیتر توربو,1405,0,تهران / ستارخان,اتوماتیک,بنزینی,مشکی,...,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,Black,Unknown,Original Paint
39519,39519,تویوتا، هایلندر هیبرید,2025 | 2.5 لیتر دو دیفرانسیل,2.5 لیتر دو دیفرانسیل,2025,0,تهران / شهید مطهری,اتوماتیک,بنزینی,مشکی,...,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,Black,Unknown,Original Paint
39539,39539,جی ای سی، ES9,2024 | پرمیوم,پرمیوم,2024,0,تهران / خیابان شیخ بهایی شمالی,اتوماتیک,پلاگین هیبرید,سفید,...,2026-10-14,0.0,True,Automatic,PHEV,Tehran,Tehran,White,Unknown,Original Paint


In [13]:
df['body_color_clean'].value_counts()

body_color_clean
White              20521
Black               9240
Gray                2357
Silver              1970
Graphite             788
Pearl White          634
Blue                 607
Titanium             520
Navy Blue            480
Brown                323
Beige                318
Red                  280
Jade Green           163
Carbon Black         154
Charcoal             143
Eggplant Purple      122
Green                120
Sour Cherry Red      112
Dolphin Gray         112
Silver Blue          106
Unknown               82
Cream                 53
Yellow                51
Olive Green           44
Atlasi                41
Lead Gray             38
Purple                36
Copper                30
Gold                  30
Orange                25
Burgundy              24
Mocha                 17
Wine Red              17
Bronze                17
Lentil Brown          16
Khaki                 15
Lilac                 14
Cherry Red            13
Camel                  6
Maroon  

In [17]:
df['body_type_org'].value_counts()

body_type_org
passenger_car    16300
crossover        11931
hatchback         7906
suv               1656
pickup            1511
van                187
coupe              103
convertible         67
Name: count, dtype: int64

In [ ]:
df['cylinder_clean'] = df['cylinder_fa_org'].str.replace(
    "سیلندر", ""
).str.strip()
df.loc[df['fuel_clean'] == 'EV', 'cylinder_clean'] = "0"
df

,listing_id,title_org,subtitle_org,trim_org,year_org,mileage_org,location_org,transmission_org,fuel_org,body_color_org,...,is_zero_kilometer,transmission_clean,fuel_clean,city_clean,province_clean,body_color_clean,inside_color_clean,body_status_clean,body_type_clean,cylinder_fa_clean
0,0,سیتروئن، C3 XR,1404 | تیپ V1,تیپ V1,1404,0,تهران / ستارخان,اتوماتیک,بنزینی,مشکی,...,True,Automatic,ICE,Tehran,Tehran,Black,Black,Original Paint,Crossover,3
1,1,لوکانو، L7,1405 | 1.6 لیتر توربو,1.6 لیتر توربو,1405,0,تهران / میرداماد,اتوماتیک,بنزینی,سفید,...,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Crossover,4
2,2,هیوندای، النترا,2025 | 1.5 لیتر,1.5 لیتر,2025,0,تهران / ستارخان,اتوماتیک,بنزینی,سفید,...,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Passenger_Car,4
3,3,چانگان، CS 55 پلاس,2024 | 1.5 لیتر توربو,1.5 لیتر توربو,2024,28000,تهران / ولنجک,اتوماتیک,بنزینی,خاکستری,...,False,Automatic,ICE,Tehran,Tehran,Gray,Black,Original Paint,Crossover,4
4,4,کی ام سی، T8,1403,دنده ای,1403,30000,تهران / میرداماد,دنده ای,بنزینی,مشکی,...,False,Manual,ICE,Tehran,Tehran,Black,Black,Original Paint,Pickup,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39656,39656,ب ام و، سری 5 سدان,2015 | 528i,528i,2015,77000,اصفهان / سعادت آباد,اتوماتیک,بنزینی,سفید,...,False,Automatic,ICE,Isfahan,Isfahan,White,Maroon,Original Paint,Passenger_Car,4
39657,39657,ب ام و، سری 5 سدان,2018 | 530i,530i,2018,49000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,مشکی,...,False,Automatic,ICE,Isfahan,Isfahan,Black,Mocha,Original Paint,Passenger_Car,4
39658,39658,نیسان، ترا,1404 | اکسکلوسیو,اکسکلوسیو,1404,0,اصفهان / سعادت آباد,اتوماتیک,بنزینی,مشکی,...,True,Automatic,ICE,Isfahan,Isfahan,Black,Maroon,Original Paint,Suv,4
39659,39659,ب ام و، سری 7,2017 | 730li,730li,2017,42000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,سفید صدفی,...,False,Automatic,ICE,Isfahan,Isfahan,Pearl White,Maroon,Original Paint,Passenger_Car,4


In [5]:
df_clean

,listing_id,brand_clean,model_clean,trim_clean,year_org,modified_date_clean,scraped_at_clean,kilometer_e-03_clean,is_zero_kilometer,transmission_clean,fuel_clean,city_clean,province_clean,body_color_clean,inside_color_clean,body_status_clean,body_type_clean,image_count_org,cylinder_fa_org,price_org
0,1,CITROEN,citroen C3 XR,Type V1,1404,2026-09-19,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,Black,Black,Original Paint,Crossover,3,3 سیلندر,3850000000
1,2,LUCANO,lucano L7,1.6L Turbo,1405,2026-10-12,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Crossover,4,4 سیلندر,0
2,3,HYUNDAI,hyundai النترا,1.5L,2025,2026-09-19,2026-10-14,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Passenger_Car,3,4 سیلندر,6700000000
3,4,CHANGAN,changan CS 55 پلاس,1.5L Turbo,2024,2026-10-12,2026-10-14,28.0,False,Automatic,ICE,Tehran,Tehran,Gray,Black,Original Paint,Crossover,6,4 سیلندر,6550000000
4,5,KMC,kmc T8,Manual,1403,2026-10-13,2026-10-14,30.0,False,Manual,ICE,Tehran,Tehran,Black,Black,Original Paint,Pickup,2,4 سیلندر,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39656,39657,BMW,bmw سری 5 سدان,528i,2015,2025-09-07,2026-10-14,77.0,False,Automatic,ICE,Isfahan,Isfahan,White,Maroon,Original Paint,Passenger_Car,8,4 سیلندر,0
39657,39658,BMW,bmw سری 5 سدان,530i,2018,2025-01-15,2026-10-14,49.0,False,Automatic,ICE,Isfahan,Isfahan,Black,Mocha,Original Paint,Passenger_Car,11,4 سیلندر,0
39658,39659,NISSAN,nissan ترا,Exclusive,1404,2025-08-30,2026-10-14,0.0,True,Automatic,ICE,Isfahan,Isfahan,Black,Maroon,Original Paint,Suv,7,4 سیلندر,0
39659,39660,BMW,bmw سری 7,730Li,2017,2024-01-23,2026-10-14,42.0,False,Automatic,ICE,Isfahan,Isfahan,Pearl White,Maroon,Original Paint,Passenger_Car,0,4 سیلندر,0
